# Day 10: Exploratory Data Analysis (EDA) & Data Cleaning Practice
## Instructor Notes & Workflow Reference

### Top 5 Datasets for EDA and Data Cleaning Practice
#### Recommended Learning Order
1. **Titanic Dataset** (Focus: Outlier detection, missing value imputation, categorical vs numerical bivariate analysis)
2. **Students Performance Dataset** (Focus: Group-wise comparisons, score distributions, categorical analysis)
3. **Mall Customers Dataset** (Focus: Customer segmentation, distribution analysis, scatter plots)
4. **Netflix Dataset** (Focus: Text cleaning, date processing, handling missing values)
5. **House Prices Dataset** (Focus: Advanced EDA, feature selection, skewness, housing trends)

---

### Standard EDA & Data Cleaning Workflow
1. **Load Dataset**: Read data from source file (CSV, Excel, etc.)
2. **Check Shape**: Verify number of rows and columns
3. **View Dataset Information**: Quick preview using `head()`, `tail()`, and `sample()`
4. **Check Data Types**: Identify numerical and categorical columns
5. **Identify Missing Values**: Check counts and percentages of nulls
6. **Handle Missing Values**: Impute or drop based on threshold and logic
7. **Check Duplicate Records**: Identify repeated rows
8. **Remove Duplicates**: Clean redundant records
9. **Generate Statistical Summary**: Compute mean, median, min, max, std dev, etc.
10. **Detect Outliers**: Using Boxplots, IQR, or Z-Score method
11. **Perform Univariate Analysis**: Analyze single variables (histograms, bar plots)
12. **Perform Bivariate Analysis**: Analyze relationships between two variables (scatter plots, crosstabs)
13. **Correlation Analysis**: Visualize numerical correlations using heatmaps
14. **Feature Engineering**: Create new variables from existing ones
15. **Draw Conclusions and Insights**: Write down key takeaways


# Titanic Dataset Analysis (Pure Matplotlib Version)
In this notebook, we follow the 15-step standard EDA workflow to explore the Titanic passenger dataset (`tested.csv`). All visualizations are built using Matplotlib and Pandas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set visual parameters for Matplotlib
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12


### Step 1: Load Dataset


In [ ]:
df = pd.read_csv("../../DataSet/titanic_data/tested.csv")
print("Dataset loaded successfully!")


### Step 2: Check Shape (Rows & Columns)


In [ ]:
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")
df.shape


### Step 3: View Dataset Information


In [ ]:
# View first 5 records
df.head()


In [ ]:
# View last 5 records
df.tail()


In [ ]:
# View a random sample of 5 records
df.sample(5)


### Step 4: Check Data Types & Columns


In [ ]:
print("Columns:", list(df.columns))
print("\nData Types:")
print(df.dtypes)


In [ ]:
# Get a complete info summary of the dataset
df.info()


### Step 5: Identify Missing Values


In [ ]:
# Count of missing values per column
missing_counts = df.isnull().sum()
# Percentage of missing values per column
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_pct
})
print("Missing Value Analysis:")
missing_df[missing_df['Missing Count'] > 0]


### Step 6: Handle Missing Values
* **Age**: 86 missing values (~20.5%). We will fill this with the **median** of the column to avoid outlier distortion.
* **Fare**: 1 missing value. We will fill it with the **median**.
* **Embarked**: 0 missing values in this file, but if any, we'd fill with the **mode**.
* **Cabin**: 327 missing values (~78.2%). Since more than 75% of the data is missing, we will **drop** this column.


In [ ]:
# 1. Fill missing Age with median
df["Age"] = df["Age"].fillna(df["Age"].median())

# 2. Fill missing Fare with median
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# 3. Fill missing Embarked with mode (if any missing)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# 4. Drop Cabin column
df.drop("Cabin", axis=1, inplace=True)

# Verify if missing values are resolved
df.isnull().sum()


### Step 7: Check Duplicate Records


In [ ]:
duplicates_count = df.duplicated().sum()
print(f"Total duplicate records found: {duplicates_count}")


### Step 8: Remove Duplicates


In [ ]:
# Ensure this step is executed in our workflow
df.drop_duplicates(inplace=True)
print("Duplicates removed! Current shape:", df.shape)


### Step 9: Generate Statistical Summary


In [ ]:
# Numerical columns statistics
df.describe()


In [ ]:
# Categorical columns statistics
df.describe(include='object')


In [ ]:
# Check unique values counts per column
df.nunique()


### Step 10: Detect Outliers (IQR Method)


In [ ]:
# Boxplot of Age before outlier removal using Matplotlib
plt.figure(figsize=(8, 4))
plt.boxplot(df["Age"], vert=False, patch_artist=True, boxprops=dict(facecolor="#5c9eb2"))
plt.title("Boxplot of Age (Before Outlier Removal)")
plt.xlabel("Age")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


In [ ]:
# Calculate IQR for Age
Q1 = df["Age"].quantile(0.25)
Q3 = df["Age"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"IQR: {IQR}")
print(f"Outlier boundaries: Lower = {lower}, Upper = {upper}")

# Identify outliers
outliers = df[(df["Age"] < lower) | (df["Age"] > upper)]
print(f"Number of Outliers: {len(outliers)}")
outliers.head()


In [ ]:
# Remove Outliers
df_cleaned = df[(df["Age"] >= lower) & (df["Age"] <= upper)].copy()
print(f"Original shape: {df.shape}, Cleaned shape: {df_cleaned.shape}")

# Verify After Outlier Removal
plt.figure(figsize=(8, 4))
plt.boxplot(df_cleaned["Age"], vert=False, patch_artist=True, boxprops=dict(facecolor="#8ebd8f"))
plt.title("Age Boxplot (After Outlier Removal)")
plt.xlabel("Age")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


### Step 11: Perform Univariate Analysis


In [ ]:
# Age distribution histogram
plt.figure(figsize=(8, 4))
plt.hist(df_cleaned["Age"], bins=20, color="purple", edgecolor="black", alpha=0.7)
plt.title("Age Distribution of Passengers")
plt.xlabel("Age")
plt.ylabel("Passenger Count")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


In [ ]:
# Survival distribution bar chart
survived_counts = df_cleaned["Survived"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(["Died (0)", "Survived (1)"], survived_counts, color=["#e74c3c", "#3498db"], edgecolor="black", width=0.6)
plt.title("Survival Counts")
plt.ylabel("Count")
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.show()

# Survival percentages
survived_pct = df_cleaned["Survived"].value_counts(normalize=True) * 100
print("Survival Percentage:")
print(f"Died: {survived_pct[0]:.2f}%")
print(f"Survived: {survived_pct[1]:.2f}%")


In [ ]:
# Gender distribution bar chart
sex_counts = df_cleaned["Sex"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(sex_counts.index, sex_counts.values, color=["#34495e", "#e67e22"], edgecolor="black", width=0.6)
plt.title("Gender Distribution")
plt.ylabel("Count")
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.show()

gender_pct = df_cleaned["Sex"].value_counts(normalize=True) * 100
print("Gender Percentage:")
print(f"Male: {gender_pct['male']:.2f}%")
print(f"Female: {gender_pct['female']:.2f}%")


### Step 12: Perform Bivariate Analysis


In [ ]:
# 1. Survival by Gender (using Pandas cross-tabulation and stacked bar plot)
crosstab_gender = pd.crosstab(df_cleaned["Sex"], df_cleaned["Survived"])
print("Gender vs Survival Crosstab:")
display(crosstab_gender)

# Plot Bivariate Gender vs Survival
crosstab_gender.plot(kind="bar", color=["#e74c3c", "#3498db"], edgecolor="black", width=0.6)
plt.title("Survival Count by Gender")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.legend(["Died", "Survived"])
plt.xticks(rotation=0)
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.show()

# Survival Rate by Gender
gender_survival_rate = pd.crosstab(df_cleaned["Sex"], df_cleaned["Survived"], normalize="index") * 100
print("Survival Rates by Gender (%):")
display(gender_survival_rate)


In [ ]:
# 2. Survival by Passenger Class (Pclass)
crosstab_class = pd.crosstab(df_cleaned["Pclass"], df_cleaned["Survived"])
print("Pclass vs Survival Crosstab:")
display(crosstab_class)

# Plot Bivariate Class vs Survival
crosstab_class.plot(kind="bar", color=["#e74c3c", "#2ecc71"], edgecolor="black", width=0.6)
plt.title("Survival Count by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Count")
plt.legend(["Died", "Survived"])
plt.xticks(rotation=0)
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.show()

# Survival Rate by Passenger Class
class_survival_rate = pd.crosstab(df_cleaned["Pclass"], df_cleaned["Survived"], normalize="index") * 100
print("Survival Rates by Class (%):")
display(class_survival_rate)


In [ ]:
# 3. Class Survival Rate using Pie Charts
d = pd.crosstab(df_cleaned["Pclass"], df_cleaned["Survived"])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#ff9999', '#66b3ff']

for i in range(1, 4):
    axes[i-1].pie(d.loc[i], labels=['Died', 'Survived'], autopct='%1.1f%%', colors=colors, startangle=90)
    axes[i-1].set_title(f"Class {i} Survival Rate")

plt.suptitle("Titanic Survival Rates by Passenger Class", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Age vs Fare Scatter Plot (Numerical vs Numerical)
plt.figure(figsize=(8, 5))
colors = df_cleaned["Survived"].map({0: "#e74c3c", 1: "#3498db"})
plt.scatter(df_cleaned["Age"], df_cleaned["Fare"], c=colors, alpha=0.7, edgecolors="black", linewidths=0.5)
plt.title("Scatter Plot: Age vs Ticket Fare (Red = Died, Blue = Survived)")
plt.xlabel("Age")
plt.ylabel("Fare")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


### Step 13: Correlation Analysis


In [ ]:
# Compute Correlation Matrix for numerical columns
corr = df_cleaned.corr(numeric_only=True)

# Generate Heatmap using Matplotlib's imshow
plt.figure(figsize=(8, 6))
plt.imshow(corr, cmap="coolwarm", interpolation="none", vmin=-1, vmax=1)
plt.colorbar(label="Correlation Coefficient")

# Set labels
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)

# Annotate correlation values inside the cells
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontweight="bold")

plt.title("Correlation Matrix Heatmap (Matplotlib)")
plt.tight_layout()
plt.show()


### Step 14: Feature Engineering


In [ ]:
# 1. Calculate Family Size
df_cleaned["FamilySize"] = df_cleaned["SibSp"] + df_cleaned["Parch"] + 1

# 2. IsAlone (1 if alone, 0 if traveling with family)
df_cleaned["IsAlone"] = np.where(df_cleaned["FamilySize"] == 1, 1, 0)

print(df_cleaned[["SibSp", "Parch", "FamilySize", "IsAlone"]].head())

# Visualize Family Size vs Survival counts
crosstab_family = pd.crosstab(df_cleaned["FamilySize"], df_cleaned["Survived"])
crosstab_family.plot(kind="bar", color=["#e74c3c", "#3498db"], edgecolor="black", width=0.7)
plt.title("Survival Counts by Family Size")
plt.xlabel("Family Size")
plt.ylabel("Count")
plt.legend(["Died", "Survived"])
plt.xticks(rotation=0)
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.show()


### Step 15: Draw Conclusions and Insights
Based on our exploratory data analysis, here are the key insights:
1. **Gender Bias in Survival**: Gender played a massive role in survival. In this test dataset, 100% of females survived, while all males died.
2. **Passenger Class Impact**: High-class passengers had a much better chance of survival. First-class passengers had a lower relative death rate than second-class and third-class passengers.
3. **Fare & Survival**: Passengers who paid higher fares (mostly first class) had a higher survival rate, as seen in the scatter plot.
4. **Family Size**: Traveling alone vs with small families showed different survival counts. Very large families had lower survival counts.
5. **Data Cleaning Outcomes**: We dropped the `Cabin` column due to >78% missing data, filled missing values in `Age` and `Fare` using medians to handle skewness, and filtered outliers in the `Age` variable successfully using the IQR method.
